In [10]:
import openai
from openai import OpenAI
import pandas as pd
df = pd.read_csv("/home/ethanliu/dementor/disguising/minimodel_responses.csv")
# df_prompts = df.sample(n=5, random_state=40)['prompt']
# df_response_4omini = df.sample(n=5, random_state=40)['gpt4omini_response']
indices = [9791, 9914, 2178, 2180, 2222] # discrepancies

df_highest_discrepancy = df.iloc[indices]
df_prompts = df_highest_discrepancy[['prompt']]
df_response_4omini = df_highest_discrepancy[['gpt4omini_response']]

print(df_prompts)
print(df_response_4omini)

                                                 prompt
9791  "Generate two reviews in 200 tokens only, in t...
9914                                      "what is LLM?
2178  "What is the best way to spend exactly 364 dol...
2180  "Sally (a girl) has 3 brothers. Each brother h...
2222  "\u4f60\u662f\u6570\u636e\u5e93\u4e13\u5bb6\uf...
                                     gpt4omini_response
9791  ```json\n{\n    "englishLanguageReview": "I ha...
9914  LLM stands for "Large Language Model." It refe...
2178  Invest the $364 in a diversified index fund to...
2180  Sally has 0 sisters. The problem states that S...
2222  要查询乘客在新年（假设是指2023年）时的年龄，我们首先需要计算他们的出生年份。新年时的年龄...


In [11]:
# System prompt for gpt 3.5 to act like gpt-4o 
system_prompt = f'''You are a helpful AI assistant. You answer the questions provided in the style defined by the example question and responses below. Note that your task is to match the style of the responses only. 

Example 1:
prompt: {df_prompts.iloc[0]}
response: {df_response_4omini.iloc[0]}

Example 2:
prompt: {df_prompts.iloc[1]}
response: {df_response_4omini.iloc[1]}

Example 3:
prompt: {df_prompts.iloc[2]}
response: {df_response_4omini.iloc[2]}

Example 4:
prompt: {df_prompts.iloc[3]}
response: {df_response_4omini.iloc[3]}

Example 5:
prompt: {df_prompts.iloc[4]}
response: {df_response_4omini.iloc[4]}

Here is the question to answer: '''

print(system_prompt)

You are a helpful AI assistant. You answer the questions provided in the style defined by the example question and responses below. Note that your task is to match the style of the responses only. 

Example 1:
prompt: prompt    "Generate two reviews in 200 tokens only, in t...
Name: 9791, dtype: object
response: gpt4omini_response    ```json\n{\n    "englishLanguageReview": "I ha...
Name: 9791, dtype: object

Example 2:
prompt: prompt    "what is LLM?
Name: 9914, dtype: object
response: gpt4omini_response    LLM stands for "Large Language Model." It refe...
Name: 9914, dtype: object

Example 3:
prompt: prompt    "What is the best way to spend exactly 364 dol...
Name: 2178, dtype: object
response: gpt4omini_response    Invest the $364 in a diversified index fund to...
Name: 2178, dtype: object

Example 4:
prompt: prompt    "Sally (a girl) has 3 brothers. Each brother h...
Name: 2180, dtype: object
response: gpt4omini_response    Sally has 0 sisters. The problem states that S...
Name: 21

In [15]:
import pandas as pd
import asyncio
import nest_asyncio
import os
from openai import OpenAI
import tiktoken  # For token counting
from dotenv import load_dotenv

# Allow nested event loops (needed for Jupyter Notebooks)
nest_asyncio.apply()

load_dotenv()  # Load environment variables from .env file
openai_api_key = os.getenv("OPENAI_API_KEY")  # Get the API key from environment variables

client = OpenAI()
encoder = tiktoken.encoding_for_model("gpt-3.5-turbo")

SAVE_PATH = "/home/ethanliu/dementor/disguising/new_minimodel_responses.csv"
batch_size = 10000  # Adjust as needed
TOKEN_LIMIT = 8192  # Max token limit for GPT-3.5-turbo

# Load dataset
df = pd.read_csv(SAVE_PATH)
total_rows = len(df)
print(f"Total rows in dataset: {total_rows}")
assert total_rows == 10000, f"Expected 10000 rows, but found {total_rows}"

async def get_batch_gpt_responses(batch):
    async def fetch(prompt):
        full_prompt = system_prompt + "\n" + prompt

        # Token check
        num_tokens = len(encoder.encode(full_prompt))
        if num_tokens > TOKEN_LIMIT:
            return "N/A"  # Skip this prompt if it's too long

        try:
            response = await asyncio.to_thread(client.chat.completions.create, 
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": full_prompt}]
            )
            return response.choices[0].message.content 
        except Exception as e:
            return f"ERROR: {str(e)}"

    return await asyncio.gather(*[fetch(prompt) for prompt in batch])

async def main():
    # Check if "gpt35_reprompted" column exists, else add it
    if "gpt35_reprompted" not in df.columns:
        df["gpt35_reprompted"] = pd.NA
        
    # Reset all existing responses to track new processing
    df["gpt35_reprompted"] = pd.NA
    processed_count = 0

    prompts = df["prompt"].tolist()
    total_batches = (len(prompts) + batch_size - 1) // batch_size
    
    print(f"Starting processing of {len(prompts)} prompts in {total_batches} batches")
    
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        print(f"Processing batch {i // batch_size + 1}/{total_batches}...")        

        try: 
            responses = await get_batch_gpt_responses(batch)
            for j, prompt in enumerate(batch):
                df.loc[df["prompt"] == prompt, "gpt35_reprompted"] = responses[j]
                processed_count += 1
            
            df.to_csv(SAVE_PATH, index=False, escapechar='\\')
            print(f"Processed {processed_count}/{total_rows} responses")

        except Exception as e:
            print(f"Error in batch {i // batch_size + 1}: {e}")
            continue
            
    print(f"Processing complete. Total responses: {processed_count}/{total_rows}")
    assert processed_count == total_rows, f"Expected {total_rows} responses, but got {processed_count}"

# Run event loop
if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.run_until_complete(main())

Total rows in dataset: 10000
Starting processing of 10000 prompts in 1 batches
Processing batch 1/1...


Processed 10000/10000 responses
Processing complete. Total responses: 10000/10000


In [8]:
SAVE_PATH = "/home/ethanliu/dementor/disguising/newest_comparison_results.csv"

import os
if os.path.exists(SAVE_PATH):
    print(f"Attempting to delete '{SAVE_PATH}' from within the kernel...")
    try:
        os.remove(SAVE_PATH)
        print(f"Successfully deleted '{SAVE_PATH}'.")
        print(f"Does it exist now? {os.path.exists(SAVE_PATH)}") # Should be False
    except OSError as e:
        print(f"Error deleting file from kernel: {e}") # Check for permission errors
else:
     print(f"Kernel confirms '{SAVE_PATH}' does not exist before attempting delete.")


Kernel confirms '/home/ethanliu/dementor/disguising/newest_comparison_results.csv' does not exist before attempting delete.


In [17]:
import pandas as pd
import asyncio
import nest_asyncio
import os
import re
from openai import OpenAI

# Allow nested event loops (needed for Jupyter Notebooks)
nest_asyncio.apply()

client = OpenAI()

# File paths and batch size
INPUT_PATH = "/home/ethanliu/dementor/disguising/new_minimodel_responses.csv"
SAVE_PATH = "/home/ethanliu/dementor/disguising/repr_comparison_results.csv"
BATCH_SIZE = 10000

# Define your comparison system prompt (customize as needed)
comparison_system_prompt = "Your system prompt here"

def parse_comparison_result(text):
    """
    Extracts the similarity score (as a float) and explanation from the model output.
    The output should begin with a number (the similarity score) followed by a period.
    """
    match = re.match(r"\s*(\d+(?:\.\d+)?)[\.\)]\s*(.*)", text)
    if match:
        score = float(match.group(1))
        explanation = match.group(2)
        return score, explanation
    else:
        return None, text

async def compare_pair(prompt, response1, response2):
    def build_prompt(r1, r2):
        return f'''Compare these two model outputs:

Response 1:
{r1}

Response 2:
{r2}

Please evaluate their similarity according to the criteria provided.
Your answer should begin with a numerical similarity score followed by a period (e.g., "2. ...").'''
    
    async def get_comparison(r1, r2):
        try:
            result = await asyncio.to_thread(client.chat.completions.create,
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": comparison_system_prompt},
                    {"role": "user", "content": build_prompt(r1, r2)}
                ]
            )
            return result.choices[0].message.content
        except Exception as e:
            return f"ERROR: {str(e)}"
    
    # Get two comparisons concurrently: one in the original order and one with swapped responses.
    result1, result2 = await asyncio.gather(
        get_comparison(response1, response2),
        get_comparison(response2, response1)
    )
    
    # Parse the similarity scores and explanations from both responses.
    score1, expl1 = parse_comparison_result(result1)
    score2, expl2 = parse_comparison_result(result2)
    
    if score1 is not None and score2 is not None:
        avg_score = (score1 + score2) / 2
        combined_expl = f"{expl1} / {expl2}"
        final_result = f"{int(round(avg_score))}. {combined_expl}"
    else:
        final_result = f"ERROR: Unable to parse similarity scores. Results: {result1} || {result2}"
    return final_result

async def compare_responses(batch):
    return await asyncio.gather(*[
        compare_pair(prompt, response1, response2)
        for prompt, response1, response2 in batch
    ])

async def main_comparison():
    # Load the input dataset.
    df = pd.read_csv(INPUT_PATH)
    
    # Load existing comparison results if available to resume processing.
    if os.path.exists(SAVE_PATH):
        existing_df = pd.read_csv(SAVE_PATH)
        processed_prompts = set(existing_df["prompt"])
        print(f"Resuming from {len(existing_df)} saved comparisons.")
    else:
        existing_df = pd.DataFrame()
        processed_prompts = set()
    
    # Prepare rows that have not yet been processed.
    all_rows = []
    for _, row in df.iterrows():
        if row["prompt"] not in processed_prompts:
            all_rows.append((row["prompt"], row["gpt35_reprompted"], row["gpt4omini_response"]))
    
    total_batches = (len(all_rows) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"Starting comparison of {len(all_rows)} unprocessed rows in {total_batches} batches.")
    
    new_data = []
    for i in range(0, len(all_rows), BATCH_SIZE):
        batch = all_rows[i:i+BATCH_SIZE]
        print(f"Processing comparison batch {i // BATCH_SIZE + 1}/{total_batches}...")
        try:
            batch_results = await compare_responses(batch)
        except Exception as e:
            print(f"Error in batch {i // BATCH_SIZE + 1}: {e}")
            batch_results = ["ERROR"] * len(batch)
        
        for (prompt, response1, response2), comparison in zip(batch, batch_results):
            new_data.append({
                "prompt": prompt,
                "gpt35_reprompted": response1,
                "gpt4omini_response": response2,
                "comparison_results": comparison
            })
        
        # Save after processing each batch by merging with existing results.
        batch_df = pd.DataFrame(new_data)
        if not existing_df.empty:
            combined_df = pd.concat([existing_df, batch_df], ignore_index=True)
        else:
            combined_df = batch_df
        combined_df.to_csv(SAVE_PATH, index=False)
        print(f"Saved {len(combined_df)} comparison results so far.")
    
    print("Comparison complete. CSV file saved successfully!")

# Run the comparison process
if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.run_until_complete(main_comparison())


Resuming from 40 saved comparisons.
Starting comparison of 9959 unprocessed rows in 1 batches.
Processing comparison batch 1/1...
Saved 9999 comparison results so far.
Comparison complete. CSV file saved successfully!


In [14]:
import pandas as pd
df = pd.read_csv("comparison_results.csv")
print(f"The dataframe has {len(df)} rows.")

The dataframe has 10000 rows.


In [13]:
import pandas as pd
import re

# File path to your CSV file containing the comparison results
FILE_PATH = "/home/ethanliu/dementor/disguising/comparison_results.csv"

def extract_score(text):
    """
    Extracts the numerical similarity score from the beginning of a comparison result.
    Expected format: "<number>. <explanation...>"
    Returns the score as a float.
    """
    match = re.match(r"^\s*(\d+(?:\.\d+)?)", text)
    if match:
        return float(match.group(1))
    else:
        return None

def main():
    # Load the CSV file into a DataFrame
    df = pd.read_csv(FILE_PATH)
    
    # Extract scores from the 'comparison_results' column using a regex
    df['score'] = df['comparison_results'].str.extract(r'^\s*(\d+(?:\.\d+)?)')[0].astype(float)
    
    # Compute standard distribution metrics
    mean_val = df['score'].mean()
    median_val = df['score'].median()
    std_val = df['score'].std()
    var_val = df['score'].var()
    min_val = df['score'].min()
    max_val = df['score'].max()
    range_val = max_val - min_val
    count_val = df['score'].count()
    
    # Print the computed statistics
    print(f"Number of rows: {count_val}")
    print(f"Mean: {mean_val:.2f}")
    print(f"Median: {median_val:.2f}")
    print(f"Standard Deviation: {std_val:.2f}")
    print(f"Variance: {var_val:.2f}")
    


if __name__ == '__main__':
    main()


Number of rows: 9993
Mean: 3.40
Median: 3.00
Standard Deviation: 1.59
Variance: 2.52


In [11]:
import pandas as pd
import asyncio
import nest_asyncio
import os
import re
from openai import OpenAI

# Allow nested event loops (needed in Jupyter Notebooks or similar environments)
nest_asyncio.apply()

client = OpenAI()

# File paths and batch size (set to 10,000 as requested)
INPUT_PATH = "/home/ethanliu/dementor/disguising/minimodel_responses.csv"
SAVE_PATH = "/home/ethanliu/dementor/disguising/new_comparison_results.csv"
BATCH_SIZE = 10000

# Define your detailed system prompt for structured output
comparison_system_prompt = '''You are an expert evaluator of AI model responses. Your task is to rate the similarity between two responses in three key dimensions:

1. Semantic Meaning (0-4 points):
   - 4: Responses convey nearly identical meaning and information
   - 3: Responses share core meaning with minor differences
   - 2: Responses have some overlapping meaning but significant differences
   - 1: Responses have minimal semantic overlap
   - 0: Responses are completely different in meaning

2. Stylistic Similarity (0-3 points):
   - 3: Responses use identical formatting (bullet points, headers, etc.)
   - 2: Responses use similar but not identical formatting
   - 1: Responses have some formatting elements in common
   - 0: Responses have completely different formatting styles

3. Response Length/Verbosity (0-3 points):
   - 3: Responses are nearly identical in length and detail level
   - 2: Responses are similar in length with minor differences
   - 1: Responses have noticeable length differences
   - 0: Responses have vastly different lengths

For each comparison, provide:
1. The distribution of scores.
2. A brief explanation of the score breakdown
3. Specific examples of similarities and differences

Example format:
4, 2, 2
Breakdown:
- Semantic Meaning: 4/4 (responses convey identical core information)
- Stylistic Similarity: 2/3 (both use bullet points but different header styles)
- Response Length: 2/3 (similar length but one has slightly more detail)

Similarities: Both responses use bullet points and cover the same key points...
Differences: Response 1 uses markdown headers while Response 2 uses plain text...
'''

def parse_comparison_result(text):
    """
    Extracts the score distribution from the beginning of the response.
    For instance, if the result begins with "4, 2, 2", this function captures those numbers.
    It then calculates the total score and returns both the scores and the remaining explanation.
    """
    # We assume the output begins with something like: "4, 2, 2"
    match = re.match(r'\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)', text)
    if match:
        scores = [int(match.group(1)), int(match.group(2)), int(match.group(3))]
        total = sum(scores)
        explanation = text[match.end():].strip()  # Everything that follows
        return scores, total, explanation
    else:
        return None, None, text

async def compare_pair(prompt, response1, response2):
    def build_prompt(r1, r2):
        return f'''Compare these two model outputs:

Response 1:
{r1}

Response 2:
{r2}

Please evaluate their similarity according to the criteria provided.
Your answer should begin with the distribution of scores for the three dimensions (e.g., "4, 2, 2") followed by the breakdown and examples.'''
    
    async def get_comparison(r1, r2):
        try:
            result = await asyncio.to_thread(
                client.chat.completions.create,
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": comparison_system_prompt},
                    {"role": "user", "content": build_prompt(r1, r2)}
                ]
            )
            return result.choices[0].message.content
        except Exception as e:
            return f"ERROR: {str(e)}"
    
    # Make two API calls concurrently: one with the original order, and one with the responses swapped.
    result1, result2 = await asyncio.gather(
        get_comparison(response1, response2),
        get_comparison(response2, response1)
    )
    
    # Parse the two results
    scores1, total1, expl1 = parse_comparison_result(result1)
    scores2, total2, expl2 = parse_comparison_result(result2)
    
    if total1 is not None and total2 is not None:
        avg_total = (total1 + total2) / 2
        combined_expl = f"Explanation from first order:\n{expl1}\n\nExplanation from swapped order:\n{expl2}"
        final_result = f"Average Total Score: {avg_total}\nCombined Breakdown:\n{combined_expl}"
    else:
        final_result = f"ERROR: Unable to parse scores. Results: {result1} || {result2}"
    return final_result

async def compare_responses(batch):
    return await asyncio.gather(*[
        compare_pair(prompt, response1, response2)
        for prompt, response1, response2 in batch
    ])

async def main_comparison():
    # Load the input dataset
    df = pd.read_csv(INPUT_PATH)
    
    # Check for existing outputs to enable resuming
    if os.path.exists(SAVE_PATH):
        existing_df = pd.read_csv(SAVE_PATH)
        processed_prompts = set(existing_df["prompt"])
        print(f"Resuming from {len(existing_df)} saved comparisons.")
    else:
        existing_df = pd.DataFrame()
        processed_prompts = set()
    
    # Prepare rows that haven't yet been processed. Each row is a tuple of (prompt, response1, response2)
    all_rows = []
    for _, row in df.iterrows():
        if row["prompt"] not in processed_prompts:
            all_rows.append((row["prompt"], row["gpt35_reprompted"], row["gpt4omini_response"]))
    
    total_batches = (len(all_rows) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"Starting comparison of {len(all_rows)} unprocessed rows in {total_batches} batch(es).")
    
    new_data = []
    for i in range(0, len(all_rows), BATCH_SIZE):
        batch = all_rows[i:i+BATCH_SIZE]
        print(f"Processing batch starting at index {i} (up to {i+BATCH_SIZE})...")
        try:
            batch_results = await compare_responses(batch)
        except Exception as e:
            print(f"Error in batch starting at index {i}: {e}")
            batch_results = ["ERROR"] * len(batch)
        
        for (prompt, response1, response2), comparison in zip(batch, batch_results):
            new_data.append({
                "prompt": prompt,
                "gpt35_reprompted": response1,
                "gpt4omini_response": response2,
                "comparison_results": comparison
            })
        
        # Save after processing each batch by merging with any existing data.
        batch_df = pd.DataFrame(new_data)
        if not existing_df.empty:
            combined_df = pd.concat([existing_df, batch_df], ignore_index=True)
        else:
            combined_df = batch_df
        combined_df.to_csv(SAVE_PATH, index=False)
        print(f"Saved {len(combined_df)} comparison results so far.")
    
    print("Comparison complete. CSV file saved successfully!")

# Run the comparison process
if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.run_until_complete(main_comparison())

Starting comparison of 10000 unprocessed rows in 1 batch(es).
Processing batch starting at index 0 (up to 10000)...
Saved 10000 comparison results so far.
Comparison complete. CSV file saved successfully!


In [18]:
# df = pd.read_csv("new_comparison_results.csv")
df = pd.read_csv("new_comparison_results.csv")
print(len(df))
# Identify rows with missing total_similarity values:


10000


In [19]:
import pandas as pd
import re
import numpy as np

def extract_section_scores(section):
    """
    Given a section (either from first order or swapped order),
    extract the three metric scores.
    
    Searches for:
      - Semantic Meaning: <num>/<max>
      - Stylistic Similarity: <num>/<max>
      - Response Length or Response Length/Verbosity: <num>/<max>
    
    Returns a tuple: (semantic, stylistic, length) as floats.
    If not found, returns (None, None, None).
    """
    sem_pattern = r"Semantic Meaning\W*(\d+)\s*/\s*\d+"
    style_pattern = r"Stylistic Similarity\W*(\d+)\s*/\s*\d+"
    length_pattern = r"Response Length(?:\s*/\s*Verbosity)?\W*(\d+)\s*/\s*\d+"
    
    sem_match = re.search(sem_pattern, section, flags=re.IGNORECASE)
    style_match = re.search(style_pattern, section, flags=re.IGNORECASE)
    length_match = re.search(length_pattern, section, flags=re.IGNORECASE)
    
    if sem_match and style_match and length_match:
        try:
            sem_score = float(sem_match.group(1))
            style_score = float(style_match.group(1))
            length_score = float(length_match.group(1))
            return sem_score, style_score, length_score
        except Exception:
            return None, None, None
    else:
        return None, None, None

def parse_combined_breakdown(text):
    """
    Given the full combined breakdown text, extract the scores from both 
    the "Explanation from first order:" and "Explanation from swapped order:" sections.
    
    The function first attempts to split the text on variations of
    "Explanation from swapped order:" (allowing for optional colon and extra whitespace).
    
    If two sections are found and successfully parsed, it averages the scores.
    Otherwise, it falls back to parsing the entire text as a single evaluation.
    
    Returns (avg_semantic, avg_stylistic, avg_length) as floats,
    or (None, None, None) if parsing fails.
    """
    # Try to split on "Explanation from swapped order" (colon optional, case-insensitive)
    parts = re.split(r"Explanation from\s+swapped order:?", text, flags=re.IGNORECASE)
    if len(parts) >= 2:
        first_order_text = parts[0]
        swapped_text = parts[1]
        
        scores1 = extract_section_scores(first_order_text)
        scores2 = extract_section_scores(swapped_text)
        if None not in scores1 and None not in scores2:
            avg_sem = (scores1[0] + scores2[0]) / 2.0
            avg_style = (scores1[1] + scores2[1]) / 2.0
            avg_length = (scores1[2] + scores2[2]) / 2.0
            return avg_sem, avg_style, avg_length
    # Fallback: Try to extract scores from the whole text assuming it only has one evaluation.
    scores = extract_section_scores(text)
    if None not in scores:
        return scores  # Duplicate the single evaluation scores
    return None, None, None

def main():
    # Load the CSV file (update the file name if needed)
    df = pd.read_csv("new_comparison_results.csv")
    print(f"Number of rows in CSV: {len(df)}")
    
    # Parse the breakdown for each row and extract averaged metric scores.
    avg_metrics = df["comparison_results"].apply(parse_combined_breakdown)
    df[['avg_semantic', 'avg_stylistic', 'avg_length']] = pd.DataFrame(avg_metrics.tolist(), index=df.index)
    
    # Compute the total similarity score by summing the three averaged metrics.
    df['total_similarity'] = df['avg_semantic'] + df['avg_stylistic'] + df['avg_length']
    
    # Report rows with successfully parsed scores (non-NaN).
    num_parsed = df['total_similarity'].count()
    print(f"Rows with successfully parsed scores: {num_parsed}")
    
    # Show descriptive statistics for the total similarity scores.
    stats = df['total_similarity'].describe()
    print("Descriptive Statistics for Total Similarity Score:")
    print(stats)

if __name__ == "__main__":
    main()


Number of rows in CSV: 10000
Rows with successfully parsed scores: 10000
Descriptive Statistics for Total Similarity Score:
count    10000.000000
mean         4.427800
std          1.977683
min          0.000000
25%          3.000000
50%          4.500000
75%          6.000000
max         10.000000
Name: total_similarity, dtype: float64
